# 01 — Lakehouse Ingestion

Loads TPC-DS CSV files from ADLS into Fabric Lakehouse Delta tables.
Creates **four schemas** (one per table configuration) so all configs
can coexist in the same Lakehouse and be queried via the SQL endpoint.

| Schema | Config | Details |
|--------|--------|---------|
| `benchmark_default` | default | No partition, no Z-order, no V-order |
| `benchmark_partitioned` | partitioned | PARTITION BY `ss_sold_date_sk` |
| `benchmark_zorder` | zorder | Z-ORDER on join/filter columns |
| `benchmark_vorder` | vorder | V-Order write optimisation enabled |

**Prerequisites**
- CSV files uploaded to the Lakehouse Files section (path: `Files/tpcds/sfXX/`)
- Notebook attached to a Spark environment with sufficient compute
- Run one section at a time to avoid memory pressure at SF1000

In [ ]:
# Parameters — adjust before running
SCALE_FACTOR = 'sf10'   # 'sf10' | 'sf100' | 'sf1000'
LAKEHOUSE_NAME = 'LH_01'
FILES_PATH = f'abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com/Files/tpcds/{SCALE_FACTOR}'

# TPC-DS tables to ingest
TABLES = [
    'store_sales', 'date_dim', 'item', 'store', 'customer',
    'customer_demographics', 'promotion', 'household_demographics'
]

# Delimiter used by dsdgen
CSV_DELIMITER = '|'

print(f'Scale factor : {SCALE_FACTOR}')
print(f'Source path  : {FILES_PATH}')

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.conf.set('spark.sql.shuffle.partitions', '200')

def read_csv(table_name: str):
    path = f'{FILES_PATH}/{table_name}.csv'
    return (
        spark.read
        .option('delimiter', CSV_DELIMITER)
        .option('header', 'false')
        .option('inferSchema', 'true')
        .csv(path)
    )

print('Spark session ready.')

## Config 1: default (no partition, no Z-order, no V-order)

In [ ]:
SCHEMA = 'benchmark_default'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}')

# Disable V-order for this config
spark.conf.set('spark.sql.parquet.vorder.enabled', 'false')

for table in TABLES:
    target = f'{SCHEMA}.{table}'
    print(f'Writing {target} ...')
    df = read_csv(table)
    df.write.format('delta').mode('overwrite').saveAsTable(target)
    print(f'  Done — {df.count():,} rows')

print('Config default complete.')

## Config 2: partitioned (PARTITION BY ss_sold_date_sk on store_sales)

In [ ]:
SCHEMA = 'benchmark_partitioned'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}')
spark.conf.set('spark.sql.parquet.vorder.enabled', 'false')

for table in TABLES:
    target = f'{SCHEMA}.{table}'
    print(f'Writing {target} ...')
    df = read_csv(table)
    writer = df.write.format('delta').mode('overwrite')
    if table == 'store_sales':
        writer = writer.partitionBy('_c0')  # ss_sold_date_sk is the first column
    writer.saveAsTable(target)
    print(f'  Done — {df.count():,} rows')

print('Config partitioned complete.')

## Config 3: zorder (OPTIMIZE + ZORDER after writing)

In [ ]:
SCHEMA = 'benchmark_zorder'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}')
spark.conf.set('spark.sql.parquet.vorder.enabled', 'false')

# Map table → Z-order column names (positional: dsdgen CSV has no header)
# Adjust column names once schema is confirmed from a sample read
ZORDER_MAP = {
    'store_sales': 'ss_item_sk, ss_store_sk',
    'item':        'i_item_sk',
    'store':       's_store_sk',
    'customer':    'c_customer_sk',
}

for table in TABLES:
    target = f'{SCHEMA}.{table}'
    print(f'Writing {target} ...')
    df = read_csv(table)
    df.write.format('delta').mode('overwrite').saveAsTable(target)
    if table in ZORDER_MAP:
        print(f'  Running OPTIMIZE ZORDER BY ({ZORDER_MAP[table]}) ...')
        spark.sql(f'OPTIMIZE {target} ZORDER BY ({ZORDER_MAP[table]})')
    print(f'  Done')

print('Config zorder complete.')

## Config 4: vorder (V-Order write optimisation enabled)

In [ ]:
SCHEMA = 'benchmark_vorder'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}')

# Enable V-order at the Spark session level
spark.conf.set('spark.sql.parquet.vorder.enabled', 'true')

for table in TABLES:
    target = f'{SCHEMA}.{table}'
    print(f'Writing {target} ...')
    df = read_csv(table)
    df.write.format('delta').mode('overwrite').saveAsTable(target)
    print(f'  Done — {df.count():,} rows')

# Reset V-order to default
spark.conf.set('spark.sql.parquet.vorder.enabled', 'false')
print('Config vorder complete.')

## Verify all schemas
Quick row-count verification across all four schemas.

In [ ]:
schemas = ['benchmark_default', 'benchmark_partitioned', 'benchmark_zorder', 'benchmark_vorder']

for schema in schemas:
    print(f'\n--- {schema} ---')
    for table in TABLES:
        count = spark.table(f'{schema}.{table}').count()
        print(f'  {table:<35} {count:>15,} rows')